# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook provides a step-by-step demonstration for loading and exploring the **A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya** using the `mlcroissant` library.

### Dataset Source
- Croissant schema: [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

This dataset comprises mental health survey data collected from Kilifi County, Kenya. It includes demographic information, psychological assessment scores (GAD-7, PHQ-9, PSQ), and other related variables.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
We'll load the metadata and records from the FAIR² Mental Health dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings("ignore")

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"Date published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Let's list the available **record sets** and their fields using their `@id`s. This helps us know what data is available, the structure, and how to reference them for extraction and processing.

> For consistent referencing, we use the `@id` attribute of each record set and field.

In [ ]:
# List record sets and fields by @id

# Each record set is a mlc.RecordSet object accessible through dataset.record_sets
record_sets = dataset.record_sets
print(f"Total record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', 'N/A')}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, data type: {field.dataType})")
    print("")

## 3. Data Extraction
We'll extract data from the main record set, referencing the appropriate `@id`. This data is loaded into a pandas DataFrame for manipulation. You can add more record sets as needed.

In [ ]:
# Use the actual RecordSet @id (as discovered above, usually of the form 'cr:<table_name>')
# For this dataset, typically there is a main survey record set.

# Find @ids for all record sets
record_set_ids = [rs.id for rs in dataset.record_sets]
print("RecordSet @ids:", record_set_ids)

# Let's extract the main survey record set. For this example, we pick the first one.
main_record_set_id = record_set_ids[0]
print(f"\nExtracting data from: {main_record_set_id}")

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Let's show fields for the main record set
print("\nColumns in main record set DataFrame:")
print(dataframes[main_record_set_id].columns.tolist())

dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's perform elementary EDA: filtering records, normalizing a numeric field, and grouping records. We use column `@id`s extracted earlier, e.g., the GAD-7 total score or PHQ-9 total score for numeric analysis.

In [ ]:
# For demonstration, manually select likely numeric and group fields by their field @id
# You can customize these by matching with the schema listing above
df = dataframes[main_record_set_id]

# For example, let's try 'gad_7_total_score' and 'gender' as field names. These could look like '@id': 'cr:gad_7_total_score', field name 'GAD-7 total score'
# For illustration, we'll infer literal column names; you should use your actual field @ids for automated reproducibility.

if 'gad_7_total_score' in df.columns:
    numeric_field_id = 'gad_7_total_score'  # Original field: '@id': 'cr:gad_7_total_score'
elif 'cr:gad_7_total_score' in df.columns:
    numeric_field_id = 'cr:gad_7_total_score'
else:
    numeric_field_id = df.columns[df.columns.str.contains('gad', case=False)][0]
threshold = 10
filtered_df = df[df[numeric_field_id].astype(float) > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()) / filtered_df[numeric_field_id].astype(float).std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field (e.g., 'gender' or '@id': 'cr:gender')
if 'gender' in df.columns:
    group_field_id = 'gender'
elif 'cr:gender' in df.columns:
    group_field_id = 'cr:gender'
else:
    group_fields = [c for c in df.columns if 'gender' in c.lower()]
    group_field_id = group_fields[0] if group_fields else df.columns[0]
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Plot the distribution of a numeric field (e.g., GAD-7 total score) by group (e.g., gender) to visualize mental health symptom prevalence.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 5))
sns.histplot(data=df, x=numeric_field_id, kde=True, bins=15, color='skyblue')
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

plt.figure(figsize=(7, 5))
if group_field_id in df.columns:
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we've loaded the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) using the `mlcroissant` library, explored the data structure referencing all key entities by their `@id`, extracted a main survey record set, and performed basic filtering, normalization, grouping, and visualization.

This demonstrates how to use Croissant schema-driven datasets for reproducible data science and analytics. For further analysis, refer to the data dictionary and schema documentation to use additional fields and integrate with downstream ML pipelines.